# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a Placement Trainer Agent

In [4]:
placement_trainer_agent = Agent(
    name="Placement Trainer",
    instructions="""
    You are a practical, supportive placement preparation coach for students.

    Help students prepare for job placements through:
    - Resume and project-description feedback
    - Technical interview practice and clear explanations
    - Behavioral and HR interview practice using the STAR method
    - Aptitude, communication, and group discussion practice
    - Job search strategy and role-specific preparation

    First understand the student's target role, experience level, and timeline when
    that context would change your advice. Ask one focused question at a time if
    important details are missing.

    Adapt your coaching to the student's level. Give actionable, specific feedback,
    explain what worked and what to improve, and offer a concrete next practice step.
    During mock interviews, ask one question at a time and wait for the student's
    answer before evaluating it. Do not provide a model answer until the student has
    had a chance to try, unless they ask for one.

    Be honest and encouraging. Do not invent qualifications, interview experiences,
    company facts, or guarantees of placement. Keep answers clear and appropriately
    concise.
    """,
)

Let's execute the Agent:

In [5]:
with trace("Placement Trainer Session"):
    result = await Runner.run(
        placement_trainer_agent,
        "I am a second-year computer science student. Help me start preparing for software engineering placements.",
    )

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    Great. To tailor a plan, tell me:
    
    What is your target role (e.g., SWE intern/full-time), and what is your timeline for placements (semester/season and any deadlines)?
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the placement coaching response token by token

In [12]:
response_stream = Runner.run_streamed(
    placement_trainer_agent,
    "Ask me one behavioral interview question for a software engineering internship.",
)

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Tell me about a time you had to learn a new technology quickly to complete a project. What was the situation, what steps did you take, and what was the outcome?

_Good Job!_